# 07 · Shuffle, Wide vs. Narrow, Broadcast Join 

**Teoria**: docs/03-transformacoes-acoes-dag.md, docs/06-persistencia-e-otimizacao.md

**Pré-requisito**: `make spark` ainda em execução.

🎯 **Objetivo**: entender na prática a diferença entre transformações **narrow** (sem shuffle) 
e **wide** (com shuffle), e como o broadcast join elimina o shuffle do lado grande.

Este laboratório coloca um join sem Shuffle (`empresas`, 50 linhas → `broadcast()`) 
lado a lado com um com muito Shuffle (`funcionarios` → `hint("merge")` força um 
SortMergeJoin de verdade). 

⚠️ **Pegadinha proposital**: `funcionarios` tem milhares de *linhas*, mas ocupa poucos 
KB em Parquet — bem abaixo do limite padrão de broadcast automático. Sem o hint 
explícito, o Catalyst faria broadcast dos dois lados, e você não veria shuffle nenhum. 
Isso é o próprio ponto do notebook: a decisão de broadcast é sobre **tamanho em bytes**, 
não sobre número de linhas.

![w:880](https://zayftfvs4d0slibc.public.blob.vercel-storage.com/shuffle_spark.png)

![w:880](https://zayftfvs4d0slibc.public.blob.vercel-storage.com/broadcast_join.png)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

In [ ]:
# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("/data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("/data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("/data/bronze/vendas")

## Broadcast join — sem Shuffle de `vendas`

📌 Procure por `BroadcastHashJoin` no plano abaixo. Depois verifique a aba 
Stages da Spark UI (http://localhost:4040) para este Job: nenhum nó `Exchange` para o 
lado grande.

🧠 **Como funciona**: o Spark copia a tabela `empresas` (50 linhas) para a memória de 
cada executor. O join acontece localmente, sem movimento de dados do lado `vendas`.

In [ ]:
from pyspark.sql.functions import sum as spark_sum, broadcast

#  Broadcast join: dica explícita broadcast() — força BroadcastHashJoin
# empresas é pequena (~50 linhas) e será copiada para cada executor
broadcast_join = sdf_vendas.join(broadcast(sdf_empresas), "id_empresa")

# explain() mostra o plano físico — procure por BroadcastHashJoin
broadcast_join.explain()

# Executa o join com groupBy para materializar o resultado
# Obs.: o count() no final força a execução
sdf = broadcast_join.groupBy("setor").agg(spark_sum("valor"))
print(f"Total: {sdf.count()}")
sdf.show()


📌 **Resultado do broadcast join**:

O plano deve mostrar `BroadcastHashJoin` sem `Exchange` no lado `vendas`. 
Isso significa ZERO shuffle de dados — apenas uma cópia pequena de `empresas` 
para cada worker.

💡 **Dica**: na aba **Stages** da Spark UI, compare o número de Stages e Tasks 
deste Job com o próximo (shuffle join). O broadcast join tipicamente tem menos Stages.

## Shuffle join — forçando um SortMergeJoin de verdade

📌 Procure por `SortMergeJoin` e dois nós `Exchange` — um para cada lado do 
join. Na Spark UI, compare a duração deste Stage contra o 
broadcast join acima.

🧠 **Como funciona**: `funcionarios` cabe tranquilamente dentro do limite padrão de 
`spark.sql.autoBroadcastJoinThreshold` (10 MB) — mesmo tendo milhares de linhas, o 
arquivo Parquet ocupa poucos KB. Sem instrução explícita, o Catalyst faria broadcast 
dele também, e o exemplo não demonstraria shuffle nenhum. Por isso usamos 
`.hint("merge")`: um **join strategy hint** que força o Spark a usar SortMergeJoin, 
ignorando o threshold de broadcast só para este join. Isso reparticiona (shuffle) 
AMBOS os DataFrames por `id_funcionario` antes de casar as linhas — muito mais caro 
em termos de rede e disco que o broadcast do exemplo anterior.

In [ ]:
# Shuffle join: hint("merge") força SortMergeJoin, ignorando o autoBroadcastJoinThreshold
# Sem o hint, o Catalyst faria broadcast de funcionarios também (arquivo pequeno em bytes)
shuffle_join = sdf_vendas.join(sdf_funcionarios.hint("merge"), "id_funcionario")

# explain() agora deve mostrar SortMergeJoin com um Exchange em cada lado
shuffle_join.explain()

# Executa com groupBy para materializar
shuffle_join.groupBy("cargo").agg(spark_sum("valor")).count()

📌 **Resultado do shuffle join**:

O plano deve mostrar `SortMergeJoin` com dois `Exchange` (um para cada lado). 
Cada `Exchange` representa um shuffle — dados sendo reescritos em disco e transferidos 
pela rede entre executores.

⚠️ **Atenção**: sem o `.hint("merge")`, o Catalyst teria escolhido `BroadcastHashJoin` 
de qualquer forma — `funcionarios` cabe bem abaixo do limite padrão de 10 MB, mesmo 
tendo milhares de linhas. É por isso que **contagem de linhas não é um bom proxy** 
para decidir se um dataset "cabe" em broadcast — o que importa é o tamanho em bytes. 
Em produção, o shuffle join pode ser 10×-100× mais lento que o broadcast join 
dependendo do volume de dados real. Na Spark UI, veja a diferença na métrica 
**Shuffle Read/Write** (MB transferidos) comparada ao broadcast join anterior.

## Reparticionando uma vez, reutilizando entre operações

🧠 **Estratégia**: se você sabe que executará várias consultas `groupBy("id_empresa")` 
em sequência, repartitionar por essa chave antecipadamente evita pagar o custo do 
Shuffle mais de uma vez.

📌 O `repartition(8, "id_empresa")` faz um shuffle único e caro, mas os groupBy 
subsequentes aproveitam a partição já alinhada — sem novo shuffle.

In [ ]:
# Reparticiona por id_empresa em 8 partições — shuffle único e proposital
# Após isso, todos os dados com o mesmo id_empresa estão na MESMA partição
vendas_por_empresa = sdf_vendas.repartition(8, "id_empresa")
vendas_por_empresa.cache()                         # Cacheia para reuso
vendas_por_empresa.count()                         # Materializa o cache

# explain() do groupBy reaproveitando o particionamento: repare que NÃO aparece
# um Exchange antes do HashAggregate — spark.sql.shuffle.partitions=8 (definido na
# SparkSession) bate com as 8 partições do repartition(), então o Catalyst detecta
# que a distribuição exigida já está satisfeita e pula o shuffle
vendas_por_empresa.groupBy("id_empresa").agg(spark_sum("valor")).explain()

# Ambos os groupBy reusam o mesmo particionamento — sem shuffle extra
vendas_por_empresa.groupBy("id_empresa").agg(spark_sum("valor")).show()
vendas_por_empresa.groupBy("id_empresa").count().show()

# Libera o cache — importante para não reter memória desnecessária
vendas_por_empresa.unpersist()

📌 **Por que isso funciona?**:

Quando você reparticiona por `id_empresa`, o Spark garante que todas as linhas com o 
mesmo `id_empresa` fiquem na mesma partição. Um `groupBy("id_empresa")` subsequente 
consegue agregar localmente, sem precisar de outro shuffle — e o `explain()` acima 
confirma isso: o plano físico vai direto para `HashAggregate` sem um `Exchange` antes.

💡 **Dica técnica**: isso é **reaproveitamento de particionamento** (o Catalyst reconhece 
que a distribuição exigida pelo `groupBy` já está satisfeita e pula o shuffle), combinado 
com `cache()` para não pagar de novo o custo de ler e reparticionar `vendas` a cada ação — 
uma técnica poderosa para pipelines que fazem múltiplas agregações pela mesma chave. Note 
que isso **não** é o mesmo que *partition pruning* (a otimização, diferente desta, que pula 
a leitura de arquivos/partições inteiras no storage com base em filtros). Repare também que 
isso só funciona porque `spark.sql.shuffle.partitions` (8, definido na SparkSession) é 
**igual** ao número de partições do `repartition(8, ...)` — se os números divergirem, o 
Catalyst não reconhece a distribuição como compatível e faz o shuffle de novo.

⚠️ **Atenção**: o `repartition()` em si é uma operação **wide** (shuffle). O benefício 
só aparece se você fizer pelo menos 2 operações após ele.